How is speech introduced in the Grimm fairy tales? Do characters *say*, *cry*, *answer*, *ask*...? And is it different for female and male characters?

In the TEI files every speech is in a `<said>` element. The verb that introduces it is in the narration around it, in the **reporting clause**:

- **after** the speech, if the sentence goes on: ‘Why, how now, good mother,’ ***said the princess***
- **before** the speech, otherwise: ***Then the young prince said,*** ‘I am not afraid.’

Our strategy:
1. split each tale into narration and speeches;
2. for each speech, take the reporting clause: the clause right after it if it continues the sentence (it starts with a lowercase letter), otherwise the clause right before it;
3. find the verbs in the clause with spaCy and count them.

## 1. Find the files and load spaCy

In [1]:
from pathlib import Path  # Tools for files and folders.
import re  # Regular expressions.
from collections import Counter  # Counts things.
import xml.etree.ElementTree as ET  # Reads XML files.
import pandas as pd  # Tables.
import spacy  # Natural language processing: finds the verbs.

# Install the English language model for spaCy. It should be already installed, but you can uncomment the next line (remove the '#' at the beginning) if needed, in case you receive an error like: Can't find model 'en_core_web_sm'.
# %pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

nlp = spacy.load("en_core_web_sm")  # The small English model.

files = sorted(Path("../data/fairy-tales").glob("[0-9]*.xml"))  # The tales (the files starting with a number).
print(len(files), "tales")

22 tales


## 2. Split a tale into narration and speeches

We read the file as a normal text and use a regular expression to find every `<said who="...">...</said>`.

`re.split` with two groups `( )` in the pattern returns a list like this:

`[narration, speaker, speech, narration, speaker, speech, ..., narration]`

so the narration is at positions 0, 3, 6..., the speakers at 1, 4, 7... and the speeches at 2, 5, 8...

In [2]:
def remove_tags(text):
    """Remove the XML tags (e.g. <persName>) and the line breaks from a text."""
    text = re.sub(r"<[^>]+>", "", text)  # Delete everything between < and >.
    return " ".join(text.split())  # Replace line breaks and multiple spaces with one space.

def split_tale(path):
    """Return three lists: the narration pieces, the speakers and the speeches of a tale."""
    xml = path.read_text(encoding="utf-8")
    body = xml[xml.index("<body>"):]  # Skip the header: we only want the text.
    parts = re.split(r'<said[^>]*who="([^"]+)"[^>]*>(.*?)</said>', body, flags=re.S)  # re.S: "." also matches line breaks.
    narration = [remove_tags(p) for p in parts[0::3]]  # Positions 0, 3, 6, ...
    speakers = parts[1::3]  # Positions 1, 4, 7, ...
    speeches = [remove_tags(p) for p in parts[2::3]]  # Positions 2, 5, 8, ...
    return narration, speakers, speeches

narration, speakers, speeches = split_tale(files[0])  # Try with the first tale.
print(len(speeches), "speeches,", len(narration), "narration pieces")  # One narration piece more: before the first and after the last speech.
print()

for i in range(4):  # The first 4 speeches with the narration around them.
    print("BEFORE: ", narration[i][-80:])  # The last 80 characters.
    print("SPEECH: ", speakers[i], speeches[i])
    print("AFTER:  ", narration[i + 1][:80])  # The first 80 characters.
    print()

8 speeches, 9 narration pieces

BEFORE:  the river; and before it swam away it lifted its head out of the water and said,
SPEECH:  #briarrose-fish ‘I know what your wish is, and it shall be fulfilled, in return for your kindness to me--you will soon have a daughter.’
AFTER:   What the little fish had foretold soon came to pass; and the queen had a little 

BEFORE:   asked his kinsmen, and nobles, and friends, and neighbours. But the queen said,
SPEECH:  #briarrose-queen ‘I will have the fairies also, that they might be kind and good to our little daughter.’
AFTER:   Now there were thirteen fairies in the kingdom; but as the king and queen had on

BEFORE:  king and queen very much, and set to work to take her revenge. So she cried out,
SPEECH:  #briarrose-thirteenth_fairy ‘The king’s daughter shall, in her fifteenth year, be wounded by a spindle, and fall down dead.’
AFTER:   Then the twelfth of the friendly fairies, who had not yet given her gift, came f

BEFORE:  ed it the door sprang

## 3. Find the reporting clause

A clause ends at a punctuation mark (`, . ; : ! ?`):
- **after**: the first clause of the following narration, but only if it starts with a lowercase letter (the sentence goes on: *‘...,’ said the princess*);
- **before**: the last clause of the preceding narration (*Then the young prince said,* ‘...’).

When a speech is interrupted (*‘Well,’ said the grandmother, ‘we will shut the door’*), the same clause is the *after* of the first half and the *before* of the second half. To count it only once, we skip a *before* that is equal to the previous *after*.

In [3]:
def clauses(narration, i):
    """Return the clause before and the clause after speech number i."""
    following = narration[i + 1]  # Narration after the speech.
    if following[:1].islower():  # Starts with a lowercase letter: the sentence goes on.
        after = re.split(r"[,.;:!?]", following)[0].strip()  # The text up to the first punctuation mark.
    else:
        after = ""

    preceding = narration[i].rstrip(" ,:")  # Narration before the speech, without the final comma or colon.
    before = re.split(r"[,.;:!?]", preceding)[-1].strip()  # The text after the last punctuation mark.
    return before, after

previous_after = ""
for i in range(len(speeches)):  # All the speeches of the first tale.
    before, after = clauses(narration, i)
    if before == previous_after:  # Same clause as the previous speech: do not count it twice.
        before = ""
    previous_after = after
    print(speakers[i], "| before:", before, "| after:", after)

#briarrose-fish | before: and before it swam away it lifted its head out of the water and said | after: 
#briarrose-queen | before: But the queen said | after: 
#briarrose-thirteenth_fairy | before: So she cried out | after: 
#briarrose-briar_rose | before:  | after: said the princess
#briarrose-briar_rose | before:  | after: 
#briarrose-old_lady | before:  | after: said the old lady
#briarrose-briar_rose | before:  | after: said the princess
#briarrose-prince | before: Then the young prince said | after: 


## 4. Find the verbs

spaCy tells us the part of speech of each word (`pos_`) and its base form (`lemma_`): *said*, *says* → *say*; *cried* → *cry*.

In [4]:
def find_verbs(text):
    """Return the base form (lemma) of the verbs in a text."""
    doc = nlp(text)
    return [token.lemma_ for token in doc if token.pos_ == "VERB"]

print(find_verbs("and a little voice cried out and said"))
print(find_verbs("thought the princess"))

['cry', 'say']
['think']


## 5. All the tales

For every speech we save: the tale, the speaker, the gender of the speaker (from the header), the two clauses and the verbs.

One speech has two speakers (Hansel and Gretel together): it gets one row per speaker.

In [5]:
rows = []  # One row per speech.

for f in files:
    # The gender of each character, from the header (as in the other notebook).
    root = ET.parse(f).getroot()
    genders = {}
    for character in root.findall(".//{http://www.tei-c.org/ns/1.0}listPerson/*"):
        genders[character.get("{http://www.w3.org/XML/1998/namespace}id")] = character.get("gender")

    narration, speakers, speeches = split_tale(f)
    previous_after = ""
    for i in range(len(speeches)):
        before, after = clauses(narration, i)
        if before == previous_after:  # Do not count the same clause twice.
            before = ""
        previous_after = after
        verbs = find_verbs(before + ". " + after)  # The verbs of both clauses (the "." keeps them apart).

        for speaker in speakers[i].split():  # Usually one speaker.
            char_id = speaker.lstrip("#")
            rows.append({
                "tale": f.stem,
                "speaker": char_id,
                "gender": genders[char_id],
                "speech": speeches[i],
                "before": before,
                "after": after,
                "verbs": verbs,
            })

df = pd.DataFrame(rows)
print(len(df), "speeches")
df[["speaker", "gender", "before", "after", "verbs"]].head(10)

688 speeches


,speaker,gender,before,after,verbs
0,briarrose-fish,unspecified,and before it swam away it lifted its head out...,,"[swam, lift, say]"
1,briarrose-queen,female,But the queen said,,[say]
2,briarrose-thirteenth_fairy,female,So she cried out,,[cry]
3,briarrose-briar_rose,female,,said the princess,[say]
4,briarrose-briar_rose,female,,,[]
5,briarrose-old_lady,female,,said the old lady,[say]
6,briarrose-briar_rose,female,,said the princess,[say]
7,briarrose-prince,male,Then the young prince said,,[say]
8,frogprince-princess,female,and said,,[say]
9,frogprince-frog,male,and said,,[say]


## 6. The most frequent verbs

In [6]:
counts = Counter()
for verbs in df["verbs"]:
    counts.update(verbs)  # Add the verbs of each speech to the count.

for verb, n in counts.most_common(25):
    print(verb, n)

say 340
cry 62
answer 36
think 24
call 16
ask 11
reply 10
go 10
come 9
sit 6
weep 4
begin 4
sing 4
take 4
look 4
knock 4
hear 3
pass 3
see 3
run 3
sigh 3
speak 2
leave 2
comfort 2
eat 2


Most verbs are speech verbs (*say*, *cry*, *answer*, *ask*, *reply*, *call*), or verbs for inner speech (*think*). But there is some noise: *go*, *come*, *sit*... come from clauses like *and she went to the door and said*. Our method finds **all** the verbs of the clause.

In [7]:
# Speeches without any verb: the speech is not introduced (e.g. in a fast dialogue), or it is the second half of an interrupted speech.
no_verb = df[df["verbs"].str.len() == 0]  # .str.len() = length of each list.
print(len(no_verb), "speeches without a verb, out of", len(df))
no_verb[["speaker", "speech"]].head(10)

172 speeches without a verb, out of 688


,speaker,speech
4,briarrose-briar_rose,‘what are you doing there?’
11,frogprince-princess,"‘what can you do for me, you nasty frog? My go..."
14,frogprince-princess,‘this silly frog is talking! He can never even...
19,frogprince-princess,"‘at the door, that lifted my ball for me out o..."
23,frogprince-frog,‘and let me sit next to you.’
27,frogprince-princess,"‘at last he is gone, and I shall be troubled w..."
30,frogprince-frog,"‘have broken his cruel charm, and now I have n..."
35,goosegirl-waiting-maid,"‘if you are thirsty, get off yourself, and sto..."
44,goosegirl-waiting-maid,"‘pray give the girl some work to do, that she ..."
48,goosegirl-waiting-maid,‘Then tell one of your slaughterers to cut off...


## 7. Female and male speakers

Are the same verbs used for female and male characters? We compare **percentages**: female characters have more speeches than male ones, so raw counts would not be fair.

For each gender: in how many of its speeches (in %) does each verb appear?

In [8]:
main_verbs = ["say", "cry", "answer", "think", "call", "ask", "reply", "weep", "sing"]  # Verbs to compare (change the list!).

table = {}
for gender in ["female", "male", "unspecified"]:
    part = df[df["gender"] == gender]  # Only the speeches of this gender.
    percentages = {}
    for verb in main_verbs:
        with_verb = part[part["verbs"].apply(lambda verbs: verb in verbs)]  # Speeches where this verb appears.
        percentages[verb] = round(len(with_verb) / len(part) * 100, 1)
    percentages["(speeches)"] = len(part)  # How many speeches in total.
    table[gender] = percentages

pd.DataFrame(table)  # Rows = verbs, columns = genders.

,female,male,unspecified
say,47.4,54.5,33.3
cry,9.5,5.5,27.1
answer,5.5,4.1,10.4
think,4.9,2.4,0.0
call,2.6,2.4,0.0
ask,1.7,1.7,0.0
reply,1.7,1.4,0.0
weep,1.1,0.0,0.0
sing,0.0,0.7,4.2
(speeches),348.0,292.0,48.0


## 8. Read the examples

Let's read some speeches introduced by a verb.

In [9]:
def show_examples(verb, gender=None, n=5):
    """Print n speeches introduced by a verb (optionally only for one gender)."""
    part = df[df["verbs"].apply(lambda verbs: verb in verbs)]
    if gender:
        part = part[part["gender"] == gender]
    print(len(part), "speeches with", verb)
    for i, row in part.head(n).iterrows():
        print("-", row["speaker"], "|", row["before"], "|", row["speech"][:80], "|", row["after"])

show_examples("cry")

62 speeches with cry
- briarrose-thirteenth_fairy | So she cried out | ‘The king’s daughter shall, in her fifteenth year, be wounded by a spindle, and  | 
- frogprince-frog | and a little voice cried out and said | ‘Open the door, my princess dear, Open the door to thy true love here! And mind  | 
- goosegirl-princess | and cried and said | ‘What will become of me?’ | 
- goosegirl-princess | but she cried | ‘Blow, breezes, blow! Let Curdken’s hat go! Blow, breezes, blow! Let him after i | 
- goosegirl-princess | and cried | ‘Falada, Falada, there thou hangest!’ | and the head answered


In [10]:
show_examples("cry", gender="female")
print()
show_examples("cry", gender="male")

33 speeches with cry
- briarrose-thirteenth_fairy | So she cried out | ‘The king’s daughter shall, in her fifteenth year, be wounded by a spindle, and  | 
- goosegirl-princess | and cried and said | ‘What will become of me?’ | 
- goosegirl-princess | but she cried | ‘Blow, breezes, blow! Let Curdken’s hat go! Blow, breezes, blow! Let him after i | 
- goosegirl-princess | and cried | ‘Falada, Falada, there thou hangest!’ | and the head answered
- goosegirl-princess | but she cried out quickly | ‘Blow, breezes, blow! Let Curdken’s hat go! Blow, breezes, blow! Let him after i | 

16 speeches with cry
- frogprince-frog | and a little voice cried out and said | ‘Open the door, my princess dear, Open the door to thy true love here! And mind  | 
- rapunzel-kings_son | he went to the tower and cried | ‘Rapunzel, Rapunzel, Let down your hair to me.’ | 
- rapunzel-kings_son | and when the king’s son came and cried | ‘Rapunzel, Rapunzel, Let down your hair to me.’ | she let the hair down
- redcap

### Future directions
- Which verbs are noise? Add a filter so that only speech verbs are counted (hint: make a list of speech verbs and keep only the verbs in that list).
- Who *cries* in these tales? Who *answers*? Look at the examples: is *cry* about tears or about a loud voice?
- Some characters are not human (the looking-glass, the dove, the winds...). Which verbs introduce their speech?
- The clause is cut at the first comma: find an example where the verb is missed (look at `no_verb`).